<a href="https://colab.research.google.com/github/kartik2006-del/efficient-finetuning-lora/blob/main/lora_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Full Fine-Tuning VS LoRA



In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
from datasets import load_dataset, Dataset
from transformers import DataCollatorForLanguageModeling

dataset = load_dataset("yahma/alpaca-cleaned", split="train")

df = dataset.to_pandas()
df = df.drop_duplicates()
dataset = Dataset.from_pandas(df, preserve_index=False)

train_test = dataset.train_test_split(test_size=0.20, seed=42)

train_data = train_test["train"]
temp_data = train_test["test"]

validation_test = temp_data.train_test_split(test_size=0.50, seed=42)

validation_data = validation_test["train"]
test_data = validation_test["test"]

print("Train:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

Train: 41404
Validation: 5176
Test: 5176


In [13]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [14]:
def format_chat(example):
    if example["input"].strip():
        user_content = example["instruction"] + "\n\nInput:\n" + example["input"]
    else:
        user_content = example["instruction"]

    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": example["output"]}
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False
        )
    }


full_ft_test_data = train_data.select(range(100))

full_ft_test_data = full_ft_test_data.map(
    format_chat
)

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )


full_ft_test_data = full_ft_test_data.map(
    tokenize_function,
    batched=True,
    remove_columns=full_ft_test_data.column_names
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print(full_ft_test_data)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 100
})


In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

full_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

full_model.train()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [16]:
total_params = sum(p.numel() for p in full_model.parameters())
trainable_params = sum(
    p.numel() for p in full_model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percentage: {100 * trainable_params / total_params:.2f}%")

Total parameters: 494,032,768
Trainable parameters: 494,032,768
Trainable percentage: 100.00%


In [17]:
import torch

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

GPU: Tesla T4
GPU memory: 14.56 GB


In [18]:
torch.cuda.reset_peak_memory_stats()

print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

Allocated: 0.92 GB
Reserved: 0.93 GB


In [21]:
import gc
import torch

del full_model
gc.collect()
torch.cuda.empty_cache()

In [22]:
from transformers import AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

full_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="auto"
)

full_model.train()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [23]:
total_params = sum(p.numel() for p in full_model.parameters())

trainable_params = sum(
    p.numel()
    for p in full_model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percentage: {100 * trainable_params / total_params:.2f}%")

Total parameters: 494,032,768
Trainable parameters: 494,032,768
Trainable percentage: 100.00%


In [24]:
import time
from transformers import TrainingArguments, Trainer

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

full_ft_args = TrainingArguments(
    output_dir="./outputs/full_ft_test",
    max_steps=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-5,
    fp16=True,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

full_ft_trainer = Trainer(
    model=full_model,
    args=full_ft_args,
    train_dataset=full_ft_test_data,
    data_collator=data_collator
)

start_time = time.time()

full_ft_trainer.train()

training_time = time.time() - start_time
peak_memory = torch.cuda.max_memory_allocated() / 1024**3

print(f"\nTraining time: {training_time:.2f} seconds")
print(f"Peak GPU memory: {peak_memory:.2f} GB")

Step,Training Loss
1,2.248624
2,1.893425
3,2.834536
4,1.921437
5,2.601707



Training time: 2.05 seconds
Peak GPU memory: 10.03 GB


In [25]:
import gc
import torch

del full_ft_trainer
del full_model

gc.collect()
torch.cuda.empty_cache()

In [28]:
!pip uninstall torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Would remove:
    /usr/local/lib/python3.13/dist-packages/torchao-0.10.0.dist-info/*
    /usr/local/lib/python3.13/dist-packages/torchao/*
Proceed (Y/n)? Y
  Successfully uninstalled torchao-0.10.0


In [29]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

lora_test_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="auto"
)

lora_test_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)

lora_test_model = get_peft_model(
    lora_test_model,
    lora_test_config
)

lora_test_model.print_trainable_parameters()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [30]:
import time
from transformers import TrainingArguments, Trainer

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

lora_test_args = TrainingArguments(
    output_dir="./outputs/lora_comparison_test",
    max_steps=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

lora_test_trainer = Trainer(
    model=lora_test_model,
    args=lora_test_args,
    train_dataset=full_ft_test_data,
    data_collator=data_collator
)

start_time = time.time()

lora_test_trainer.train()

lora_training_time = time.time() - start_time
lora_peak_memory = torch.cuda.max_memory_allocated() / 1024**3

print(f"\nLoRA training time: {lora_training_time:.2f} seconds")
print(f"LoRA peak GPU memory: {lora_peak_memory:.2f} GB")

Step,Training Loss
1,2.248624
2,1.893425
3,2.624464
4,1.832974
5,3.231456



LoRA training time: 2.00 seconds
LoRA peak GPU memory: 9.80 GB


In [31]:
# Full Fine-Tuning parameter storage
full_ft_params = 494_032_768

# LoRA trainable parameters
lora_params = 1_081_344

# FP32 = 4 bytes per parameter
full_ft_fp32_gb = (full_ft_params * 4) / (1024**3)

# FP16 = 2 bytes per parameter
full_ft_fp16_gb = (full_ft_params * 2) / (1024**3)

# LoRA adapter in FP32 / FP16
lora_fp32_mb = (lora_params * 4) / (1024**2)
lora_fp16_mb = (lora_params * 2) / (1024**2)

print(f"Full FT weights (FP32): {full_ft_fp32_gb:.2f} GB")
print(f"Full FT weights (FP16): {full_ft_fp16_gb:.2f} GB")
print(f"LoRA adapter (FP32): {lora_fp32_mb:.2f} MB")
print(f"LoRA adapter (FP16): {lora_fp16_mb:.2f} MB")

Full FT weights (FP32): 1.84 GB
Full FT weights (FP16): 0.92 GB
LoRA adapter (FP32): 4.12 MB
LoRA adapter (FP16): 2.06 MB
